# Skripsi: Integrasi Confident Learning dan CORN pada IndoBERT
## Pilot Study — Deteksi Label Noise pada Prediksi Rating Ulasan Google Play Store

**Ryan Besto Saragih — NIM 23051204205**
Pembimbing: Anita Qoiriah, S.Kom., M.Kom.

Notebook ini menjalankan pipeline lengkap pilot study sesuai Bab 1 (Subbab 1.2) dan
Bab 3 proposal: scraping → praproses → ablasi proxy classifier (P1-P4) →
deteksi noise → validasi manusia → training 6 skenario model → uji signifikansi
statistik (4 hipotesis pre-registered).

**Struktur data**: satu skema tunggal (tidak ada versioning) — `all_reviews_master.csv`
dikunci setelah scraping selesai dan dipakai sebagai sumber kebenaran tunggal untuk
seluruh eksperimen berikutnya, sesuai Batasan Masalah Bab 1.6.

**Cara pakai**: jalankan cell berurutan dari atas ke bawah. Cell scraping (Cell 3)
otomatis dilewati kalau data sudah ada — jangan hapus `all_reviews_master.csv`
kecuali memang berniat scraping ulang dari nol.

In [1]:
# ==========================================================
# CELL 2: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/skripsi-corn-label-noise"
REPO_DIR = "/content/skripsi-corn-label-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Repo sudah ada, menarik update terbaru...
Already up to date.

✅ Setup selesai. Working dir: /content/skripsi-corn-label-noise


## Instalasi Dependencies

Termasuk `py3langid` untuk filter bahasa saat scraping. Pastikan `requirements.txt`
di repo sudah memuat: `pandas`, `numpy`, `scikit-learn`, `torch`, `transformers`,
`cleanlab`, `coral-pytorch`, `statsmodels`, `scipy`, `google-play-scraper`,
`py3langid`.

In [2]:
# ==========================================================
# CELL 4: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

✅ Dependencies terpasang.


## Import Umum
Dipakai berulang di banyak cell berikutnya — diletakkan sekali di sini supaya
tidak diulang-ulang di tiap cell.

In [3]:
# ==========================================================
# CELL 6: IMPORT UMUM
# ==========================================================
import os
import pandas as pd
import numpy as np

from src import config
print(f"📌 Proxy default aktif: [{config.PROXY_ID}] {config.PROXY_NAME}")
print(f"📁 Drive root: {config.DRIVE_ROOT}")

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
📌 Proxy default aktif: [3] finetuned_corn
📁 Drive root: /content/drive/MyDrive/SKRIPSI_CORN


## Scraping Google Play Store

Scraping dengan filter bahasa Indonesia inline (`py3langid`), target per rating
sudah dinaikkan untuk mengantisipasi susut data sepanjang pipeline (praproses →
split → K-Fold → pruning), dan dilengkapi laporan kualitas data (distribusi
tanggal per app/rating untuk cek confound temporal, distribusi panjang teks,
deteksi teks duplikat) yang otomatis tersimpan di `results/` untuk dilampirkan
di Bab III dan Bab IV.

⚠️ **JALANKAN SEKALI SAJA SEUMUR PROYEK.** Setelah selesai, `all_reviews_master.csv`
dikunci dan dipakai sebagai sumber kebenaran tunggal. Cell ini **otomatis dilewati**
kalau file tersebut sudah ada — hapus manual hanya kalau memang sengaja ingin
scraping ulang dari nol (dan sadar semua eksperimen berikutnya perlu diulang).

Estimasi waktu: bisa 1-3 jam tergantung koneksi dan seberapa jauh Google Play
Store perlu ditelusuri untuk rating 2/3 yang jarang muncul.

In [4]:
# ==========================================================
# CELL 8: SCRAPING GOOGLE PLAY STORE (AUTO-SKIP KALAU SUDAH ADA)
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
    df_check = pd.read_csv(config.RAW_DATA_FILE)
    print(f"   Baris tersedia: {len(df_check)}")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

 GOOGLE PLAY SCRAPER — STRATIFIED PER RATING + FILTER BAHASA 

📡 SeaBank | rating=1 | target=1200 (setelah filter bahasa)
   +200 diperiksa, 167/1200 lolos filter bahasa (dibuang bahasa lain: 33)
   +200 diperiksa, 329/1200 lolos filter bahasa (dibuang bahasa lain: 71)
   +200 diperiksa, 494/1200 lolos filter bahasa (dibuang bahasa lain: 106)
   +200 diperiksa, 667/1200 lolos filter bahasa (dibuang bahasa lain: 133)
   💾 Checkpoint tersimpan (667 baris)
   +200 diperiksa, 848/1200 lolos filter bahasa (dibuang bahasa lain: 152)
   +200 diperiksa, 1011/1200 lolos filter bahasa (dibuang bahasa lain: 189)
   +200 diperiksa, 1175/1200 lolos filter bahasa (dibuang bahasa lain: 225)
   💾 Checkpoint tersimpan (1175 baris)
   +200 diperiksa, 1342/1200 lolos filter bahasa (dibuang bahasa lain: 258)
✅ Selesai: SeaBank rating=1 -> 1342 ulasan bersih [PENUH] (diperiksa: 1600, dibuang-bahasa: 258, terlalu-pendek-utk-deteksi: 13)

📡 SeaBank | rating=2 | target=900 (setelah filter bahasa)
   +200 dipe

## Kunci Korpus + EDA Awal (SEBELUM Praproses)
Dijalankan sekali di atas korpus mentah: dedup review_id, kunci sumber
kebenaran tunggal (manifest + SHA-256, Batasan Masalah butir 7), dan
seluruh angka deskripsi data Subbab III.B (filter bahasa, distribusi
tanggal, panjang teks, duplikat, konflik teks-rating, cek overlap split).

In [5]:
# ==========================================================
# CELL 9: KUNCI KORPUS + EDA AWAL
# ==========================================================
from scripts.eda_dataset import main as run_eda
run_eda()

 EDA DAN PENGUNCIAN DATASET 

 0. MUAT DAN KUNCI KORPUS MENTAH 
Baris mentah dibaca            : 16740
Setelah dedup review_id       : 16740 (dibuang 0)
Setelah buang teks kosong      : 16740 (dibuang 0)
Rentang tanggal                : 2025-08-31 s.d. 2026-09-07

 1. DISTRIBUSI RATING 
           n  persen
rating              
1       3779   22.57
2       2976   17.78
3       2995   17.89
4       3176   18.97
5       3814   22.78
   -> /content/drive/MyDrive/SKRIPSI_CORN/results/eda/01_distribusi_rating_keseluruhan.csv

Per aplikasi:
rating         1     2     3     4     5
source_app                              
Gojek       1225   917   901  1035  1230
SeaBank     1342   994  1045  1088  1329
Tokopedia   1212  1065  1049  1053  1255
   -> /content/drive/MyDrive/SKRIPSI_CORN/results/eda/01_distribusi_rating_per_aplikasi.csv
   -> /content/drive/MyDrive/SKRIPSI_CORN/results/eda/01_distribusi_rating_per_aplikasi.png

Catatan Bab IV: distribusi ini hasil kuota per rating, BUKAN
distribu

## Praproses Teks

Case folding, penghapusan URL/username, translasi emoji ke sentimen (kamus
diperluas), normalisasi elongasi huruf, normalisasi slang (Kamus Alay + kamus
domain, dengan perbaikan pencocokan token bertanda-baca), resolusi konflik
teks-identik-rating-berbeda (voting mayoritas), dan penghapusan duplikat murni.

Preprocessing dijaga minimal (tanpa stemming/stopword removal) sesuai rekomendasi
untuk model berbasis BERT (Wilie dkk. 2020; Koto dkk. 2020).

⚠️ **JALANKAN SEKALI SAJA.** Split train/test yang dihasilkan (`split_train_raw.csv`
/ `split_test.csv`) dikunci dan dipakai di seluruh eksperimen berikutnya, termasuk
pilot study proxy P1-P4. Cell ini otomatis dilewati kalau split sudah ada.

In [6]:
# ==========================================================
# CELL 10: PREPROCESSING + SPLIT 70:10:20 GROUPED-STRATIFIED
# ==========================================================
from src.preprocess import run_preprocessing
from src.data_split import create_splits
from src import config

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.VAL_FILE) and os.path.exists(config.TEST_FILE):
    print("✅ Split train/val/test sudah ada -- dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_val = pd.read_csv(config.VAL_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
else:
    print("=" * 60); print(" PREPROCESSING "); print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT 70:10:20 (grouped berdasarkan teks, stratified rating, seed=42) ")
    print("=" * 60)
    df_train, df_val, df_test = create_splits(config.CLEAN_TEXT_FILE, seed=42)

⬇️  Mengunduh Kamus Alay (Salsabila dkk., 2018) dari https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv ...
✅ Tersimpan di cache: /content/drive/MyDrive/SKRIPSI_CORN/lexicon/slang_base.csv
📖 Kamus slang dasar: 15006 entri (Salsabila dkk., 2018)
 PREPROCESSING 
📥 Membaca data mentah dari: /content/drive/MyDrive/SKRIPSI_CORN/data/raw/all_reviews_master.csv
📏 Jumlah data awal: 16740 baris
🧹 Cleaning teks (lowercase, URL/tag, emoji, elongasi, slang)...
📊 Cakupan kamus slang (estimasi dari raw text): 29410/253442 kata (11.6%)
📏 Distribusi panjang teks: median=10 kata, ulasan sangat pendek (<=3 kata): 24.40%
🔍 Teks identik, rating berbeda: 151 kasus
   -> Resolusi (voting mayoritas) BELUM dieksekusi di sini. Sesuai Subbab
      III.D butir 8, resolusi dijalankan di clean.py SETELAH deteksi
      Confident Learning, agar baris ini turut dianalisis CL sebagai kandidat noise.
✅ Setelah dibersihkan: 14221 baris (terbuang: 2519)


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✂️  Ulasan yang token-nya terpotong (> 128 token): 0.06%
💾 Disimpan di: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean.csv

 SPLIT 70:10:20 (grouped berdasarkan teks, stratified rating, seed=42) 
📦 Jumlah teks unik: 13979 (dari 14221 baris)
✅ Train: 9949 baris (70.0%)
✅ Val  : 1416 baris (10.0%)
✅ Test : 2856 baris (20.1%)

📎 Baris yang terlibat pengelompokan teks duplikat: 393 (2.76%) -- dijamin satu sisi split (Subbab III.E).
   Distribusi rating Train: {1: 0.262, 2: 0.201, 3: 0.188, 4: 0.169, 5: 0.18}
   Distribusi rating Val: {1: 0.263, 2: 0.201, 3: 0.186, 4: 0.167, 5: 0.182}
   Distribusi rating Test: {1: 0.261, 2: 0.201, 3: 0.188, 4: 0.169, 5: 0.182}


## Pilot Study — Ablasi Proxy Classifier (P1-P4)

Menjalankan Confident Learning untuk 4 dari 5 tahap ablasi proxy classifier
(Tabel 3.2, Bab 3): P1 (CLS embedding beku + LR), P2 (mean-pooling beku + LR),
P3 (IndoBERT fine-tuned, Cross-Entropy), P4 (IndoBERT fine-tuned, CORN — proxy
final). P5 (fusi sentimen eksternal) dijalankan terpisah di Cell berikutnya
karena butuh model sentimen tambahan dan bukan bagian pilot study Bab 1.

Setiap proxy sekarang: (a) memakai skema K-Fold yang identik (shuffle=True,
seed=42) untuk P1/P2 maupun P3/P4, (b) dikalibrasi lewat temperature scaling
untuk P3/P4, (c) metode filter cleanlab dipilih otomatis lewat kriteria
eksplisit (paling dekat dengan estimasi teoretis `num_label_issues()`), bukan
hardcode.

Hasilnya terkumpul di `results/proxy_ablation_table.csv` — inilah **Tabel 1.1**
pilot study di Bab 1 proposal.

⏱️ P3 dan P4 butuh fine-tuning K-Fold (5 fold × 3 epoch masing-masing) — cell
ini bisa memakan waktu cukup lama.

In [7]:
# ==========================================================
# CELL 12: PILOT STUDY -- ABLASI PROXY P1-P4
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

PILOT_PROXY_IDS = [0, 1, 2, 3]  # P5 (fusion, id 4) dijalankan terpisah di Cell 14

for pid in PILOT_PROXY_IDS:
    print(f"\n{'='*70}\n PILOT STUDY -- PROXY_ID = {pid} \n{'='*70}")
    config.set_proxy(pid)
    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Pilot study (P1-P4) selesai.")
pilot_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(pilot_table)


 PILOT STUDY -- PROXY_ID = 0 
📌 Proxy aktif: [0] frozen_cls_lr — CLS embedding beku + Logistic Regression (P1)
 CONFIDENT LEARNING — proxy aktif: [0] frozen_cls_lr 
📥 Memuat 9949 baris data train.

🧮 Menghitung OOF pred_probs — proxy [0] frozen_cls_lr


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5087
      Epoch 2/3 - Loss: 0.4437
      Epoch 3/3 - Loss: 0.3754
   🌡️  Temperature scaling [finetuned_corn]: T=1.1666 (NLL 1.3101 -> 1.2867)
   [Proxy finetuned_corn] Fold 5/5 (train=7960, val=1989)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5069
      Epoch 2/3 - Loss: 0.4423
      Epoch 3/3 - Loss: 0.3715
   🌡️  Temperature scaling [finetuned_corn]: T=1.1582 (NLL 1.3178 -> 1.2959)
   🌡️  Temperature per fold [finetuned_corn]: ['1.213', '1.156', '1.217', '1.167', '1.158'] (mean=1.182)

📐 Kualitas proxy [finetuned_corn]:
   Exact Accuracy : 0.4596
   MAE            : 0.7939
   Off-by-1 Acc   : 0.8142
   QWK            : 0.6713

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 4732 baris diflag (47.56%)
   'prune_by_noise_rate': 4236 baris diflag (42.58%)

🎯 Pemilihan metode filter (kriteria: paling dekat dengan estimasi teoretis terkalibrasi):
   Estimasi teoretis (num_label_issues, off_diagonal_calibrated) : 5441 baris
   'confident_learning': 4732 baris (selisih: 709) <-- DIPILIH
   'prune_by_noise_rate': 4236 baris (selisih: 1205)

✅ Deteksi selesai (metode terpilih: confident_learning).

📊 Distribusi rating_diff pada baris noise (sebelum resolusi):
rating_diff
1    3093
2    1198
3 

,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise,selected_filter_method,n_issues_confident_learning,n_issues_prune_by_noise_rate,n_issues_estimated_theoretical
0,0,frozen_cls_lr,CLS embedding beku + Logistic Regression (P1),0.428988,0.947130,0.744999,0.610804,46.014675,confident_learning,4578,3950,5752
1,1,frozen_meanpool_lr,Mean-pooling embedding beku + Logistic Regress...,0.429289,0.959393,0.740275,0.599542,46.235803,confident_learning,4600,3954,5724
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.454619,0.785808,0.820384,0.671313,48.818977,confident_learning,4857,4372,5506
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.459644,0.793949,0.814152,0.671262,47.562569,confident_learning,4732,4236,5441


## Kunci Proxy Final (P4) + Ekspor Sample Validasi Manusia

Menjalankan ulang Confident Learning dengan proxy final terkunci (P4/CORN,
sesuai skema integrasi ketat Bab 1.1) — akan memuat dari cache kalau sudah
pernah dihitung di Cell 12, jadi cepat. Ini menghasilkan sample 50 baris
(stratified by rating_diff) untuk validasi manusia.

⚠️ **BERHENTI setelah cell ini** — isi dulu `human_verdict` secara manual
sebelum lanjut ke Cell 17.

In [8]:
# ==========================================================
# CELL 16: KUNCI PROXY FINAL (P4) -- WAJIB SEBELUM VALIDASI MANUSIA
# ==========================================================
from src import config

config.set_proxy(3)
print(f"🔒 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

print("ℹ️  Resolusi konflik teks-identik-rating-beda sekarang otomatis dieksekusi")
print("   di dalam run_confident_learning() SETELAH deteksi CL (Subbab III.D butir 8).")

print("\n⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 17 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan Cell 17 untuk cek kelengkapan + hitung agreement rate.")

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
🔒 Proxy final terkunci: [3] finetuned_corn
 CONFIDENT LEARNING — proxy aktif: [3] finetuned_corn 
📥 Memuat 9949 baris data train.

🧮 Menghitung OOF pred_probs — proxy [3] finetuned_corn
⚡ Memuat cache proxy [finetuned_corn] ...

📐 Kualitas proxy [finetuned_corn]:
   Exact Accuracy : 0.4596
   MAE            : 0.7939
   Off-by-1 Acc   : 0.8142
   QWK            : 0.6713

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 4732 baris diflag (47.56%)
   'prune_by_noise_rate': 4236 baris diflag (42.58%)

🎯 Pemilihan metode filter (kriteria: paling dekat dengan estimasi teoretis terkalibrasi):
   Estimasi teoretis (num_label_issues, off_diagonal_calibrated) : 5441 baris
   'confident_learning': 4732 baris (selisih: 709) <-- DIPILIH
   'prune_by_noise_rate': 4236 baris (selisih: 1205)

✅ Deteksi selesai (metode terpilih: confident_learning).

📊 Distribusi rating_diff pada baris noise (sebe

## Hitung Agreement Rate Validasi Manusia

Menjalankan **setelah** kamu selesai mengisi `human_verdict` secara manual di
file yang disebut Cell 16. Melaporkan agreement rate keseluruhan dan per-bin
`rating_diff` — breakdown per-bin ini penting sebagai bukti kualitatif
pendukung/penolak asumsi ambang severity-aware pruning (Subbab 3.7).

In [9]:
from src import config
import pandas as pd

df_check = pd.read_csv(config.HUMAN_VALIDATION_FILE, sep=None, engine="python")
print(df_check.shape)
print(df_check["human_verdict"].value_counts(dropna=False))

(100, 7)
human_verdict
not_noise    60
noise        26
ambiguous    14
Name: count, dtype: int64


In [10]:
# ==========================================================
# CELL 18: HITUNG AGREEMENT VALIDASI MANUSIA
# ==========================================================
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 19 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

   (file dibaca dengan delimiter ',')
 HASIL VALIDASI MANUSIA vs CLEANLAB 
Total sample direview : 100
Setuju (memang noise) : 26 (26.0%)
Ambigu                : 14 (14.0%)
Tidak setuju          : 60 (60.0%)

📊 Agreement rate per rating_diff:
   diff=1: n=55 | noise=10.9% | not_noise=81.8% | ambiguous=7.3%
   diff=2: n=15 | noise=6.7% | not_noise=53.3% | ambiguous=40.0%
   diff=3: n=15 | noise=53.3% | not_noise=26.7% | ambiguous=20.0%
   diff=4: n=15 | noise=73.3% | not_noise=20.0% | ambiguous=6.7%
------------------------------------------------------------
Acuan pembanding (Northcutt et al., 2021, ImageNet): ~58% sample
yang direview terbukti benar-benar issue -- acuan wajar, bukan standar mutlak.

💾 Hasil lengkap -> /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_result__finetuned_corn.csv

✅ Validasi manusia lengkap. Siap lanjut ke Cell 19 (training).


## Training 6 Skenario × 3 Seed

Melatih 6 skenario (kombinasi 3 varian data × 2 fungsi loss): M1-M3 (raw/
hard-prune/severity-prune × Cross-Entropy), M4-M6 (raw/hard-prune/severity-prune
× CORN). Setiap skenario dilatih pada 3 seed (42, 123, 2024) dengan
mixed-precision training dan early stopping (berhenti kalau Val MAE tidak
membaik 3 epoch berturut-turut).

Auto-resume: kalau sesi Colab terputus, jalankan ulang cell ini — training
akan melanjutkan dari skenario/seed terakhir yang belum selesai, bukan mulai
dari nol.

⏱️ Ini cell paling lama di seluruh notebook — bisa beberapa jam tergantung GPU
yang dialokasikan Colab.

In [11]:
# ==========================================================
# CELL 20: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

print(f"🔧 Training M1-M6 akan pakai backbone: {config.PRETRAINED_MODEL_NAME} | "
      f"proxy final: {config.PROXY_NAME}")

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_RESOLVED_FILE, "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE, "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE, "loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_RESOLVED_FILE, "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE, "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE, "loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(" 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆")
print("=" * 100)
display(df_final)

🔧 Training M1-M6 akan pakai backbone: indobenchmark/indobert-base-p1 | proxy final: finetuned_corn
🆕 Memulai training dari awal...

🔥 6 SKENARIO x 3 SEED 🔥


🚀 M1_Baseline_CE | Seed: 42 | Loss: CE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/10 - Loss: 1.3595 - Val MAE: 0.8121 | QWK: 0.6829 | Off-by-1: 0.7994 | Acc: 0.4597
Epoch 2/10 - Loss: 1.1865 - Val MAE: 0.7768 | QWK: 0.7000 | Off-by-1: 0.8164 | Acc: 0.4647
Epoch 3/10 - Loss: 1.0140 - Val MAE: 0.7910 | QWK: 0.6749 | Off-by-1: 0.8143 | Acc: 0.4689
Epoch 4/10 - Loss: 0.7815 - Val MAE: 0.7747 | QWK: 0.6803 | Off-by-1: 0.8305 | Acc: 0.4386
Epoch 5/10 - Loss: 0.5031 - Val MAE: 0.7726 | QWK: 0.6763 | Off-by-1: 0.8475 | Acc: 0.4223
Epoch 6/10 - Loss: 0.2976 - Val MAE: 0.8023 | QWK: 0.6612 | Off-by-1: 0.8227 | Acc: 0.4287
Epoch 7/10 - Loss: 0.1710 - Val MAE: 0.7846 | QWK: 0.6671 | Off-by-1: 0.8291 | Acc: 0.4294
Epoch 8/10 - Loss: 0.0965 - Val MAE: 0.8037 | QWK: 0.6544 | Off-by-1: 0.8305 | Acc: 0.4181
   ⏹ Early stopping di epoch 8 (Val MAE tidak membaik 3x)
🏆 Test (checkpoint QWK terbaik=0.7000, tie-break MAE=0.7768): MAE=0.7574 | RMSE=1.1663 | Acc=0.4702 | Off-by-1=0.8323 | QWK=0.7094

🚀 M1_Baseline_CE | Seed: 123 | Loss: CE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/10 - Loss: 1.3651 - Val MAE: 0.8079 | QWK: 0.6730 | Off-by-1: 0.7938 | Acc: 0.4675
Epoch 2/10 - Loss: 1.1881 - Val MAE: 0.8164 | QWK: 0.6757 | Off-by-1: 0.7903 | Acc: 0.4696
Epoch 3/10 - Loss: 1.0087 - Val MAE: 0.7839 | QWK: 0.6728 | Off-by-1: 0.8256 | Acc: 0.4336
Epoch 4/10 - Loss: 0.7550 - Val MAE: 0.7959 | QWK: 0.6689 | Off-by-1: 0.8326 | Acc: 0.4258
Epoch 5/10 - Loss: 0.4706 - Val MAE: 0.7966 | QWK: 0.6682 | Off-by-1: 0.8107 | Acc: 0.4393
Epoch 6/10 - Loss: 0.2666 - Val MAE: 0.8143 | QWK: 0.6564 | Off-by-1: 0.8150 | Acc: 0.4103
   ⏹ Early stopping di epoch 6 (Val MAE tidak membaik 3x)
🏆 Test (checkpoint QWK terbaik=0.6757, tie-break MAE=0.8164): MAE=0.7693 | RMSE=1.1951 | Acc=0.4811 | Off-by-1=0.8141 | QWK=0.7101

🚀 M1_Baseline_CE | Seed: 2024 | Loss: CE


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/10 - Loss: 0.4966 - Val MAE: 0.8121 | QWK: 0.6841 | Off-by-1: 0.8001 | Acc: 0.4470
Epoch 2/10 - Loss: 0.3799 - Val MAE: 0.7422 | QWK: 0.7104 | Off-by-1: 0.8496 | Acc: 0.4513
Epoch 3/10 - Loss: 0.3031 - Val MAE: 0.7408 | QWK: 0.7004 | Off-by-1: 0.8468 | Acc: 0.4463
Epoch 4/10 - Loss: 0.2293 - Val MAE: 0.7352 | QWK: 0.7115 | Off-by-1: 0.8524 | Acc: 0.4470
Epoch 5/10 - Loss: 0.1562 - Val MAE: 0.7373 | QWK: 0.7029 | Off-by-1: 0.8588 | Acc: 0.4393
Epoch 6/10 - Loss: 0.0979 - Val MAE: 0.7564 | QWK: 0.6896 | Off-by-1: 0.8517 | Acc: 0.4308
Epoch 7/10 - Loss: 0.0645 - Val MAE: 0.7634 | QWK: 0.6886 | Off-by-1: 0.8538 | Acc: 0.4230
   ⏹ Early stopping di epoch 7 (Val MAE tidak membaik 3x)
🏆 Test (checkpoint QWK terbaik=0.7115, tie-break MAE=0.7352): MAE=0.7286 | RMSE=1.0967 | Acc=0.4552 | Off-by-1=0.8599 | QWK=0.7079

🚀 M6_CleanedSevere_CORN | Seed: 2024 | Loss: CORN


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/10 - Loss: 0.4941 - Val MAE: 0.7500 | QWK: 0.7071 | Off-by-1: 0.8439 | Acc: 0.4506
Epoch 2/10 - Loss: 0.3814 - Val MAE: 0.7401 | QWK: 0.7078 | Off-by-1: 0.8432 | Acc: 0.4612
Epoch 3/10 - Loss: 0.3056 - Val MAE: 0.7458 | QWK: 0.7043 | Off-by-1: 0.8453 | Acc: 0.4470
Epoch 4/10 - Loss: 0.2222 - Val MAE: 0.7599 | QWK: 0.6919 | Off-by-1: 0.8489 | Acc: 0.4315
Epoch 5/10 - Loss: 0.1474 - Val MAE: 0.7479 | QWK: 0.7021 | Off-by-1: 0.8468 | Acc: 0.4435
   ⏹ Early stopping di epoch 5 (Val MAE tidak membaik 3x)
🏆 Test (checkpoint QWK terbaik=0.7078, tie-break MAE=0.7401): MAE=0.7188 | RMSE=1.0864 | Acc=0.4629 | Off-by-1=0.8589 | QWK=0.7252

 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M4_Baseline_CORN,0.7006 ± 0.0061,1.0615 ± 0.0088,0.4659 ± 0.0034,0.8718 ± 0.0032,0.7281 ± 0.0014
1,M6_CleanedSevere_CORN,0.7225 ± 0.0044,1.0864 ± 0.0084,0.4573 ± 0.0040,0.8611 ± 0.0024,0.7148 ± 0.0075
2,M5_CleanedHard_CORN,0.7233 ± 0.0072,1.0987 ± 0.0079,0.4653 ± 0.0065,0.8559 ± 0.0030,0.7171 ± 0.0027
3,M3_CleanedSevere_CE,0.7271 ± 0.0117,1.1093 ± 0.0113,0.4672 ± 0.0140,0.8525 ± 0.0040,0.7137 ± 0.0048
4,M2_CleanedHard_CE,0.7293 ± 0.0075,1.1165 ± 0.0091,0.4680 ± 0.0034,0.8527 ± 0.0022,0.7133 ± 0.0023
5,M1_Baseline_CE,0.7612 ± 0.0057,1.1621 ± 0.0288,0.4643 ± 0.0167,0.8310 ± 0.0133,0.7058 ± 0.0056


## Uji Signifikansi — Wilcoxon + Holm-Bonferroni (5 Hipotesis) + CI Bootstrap QWK
H1: CORN vs CE pada data mentah (M4 vs M1)
H2: severity-aware vs baseline pada CORN (M6 vs M4)
H3: hard-prune vs baseline pada CORN (M5 vs M4)
H4: severity-aware vs hard-prune langsung (M6 vs M5)
H5: severity-aware vs baseline pada CE (M3 vs M1) — apakah manfaat cleaning
    selektif spesifik pada arsitektur ordinal, atau berlaku umum lintas fungsi loss
QWK adalah metrik keputusan utama tapi tidak bisa diuji Wilcoxon (dihitung
atas matriks konfusi agregat) — dilaporkan via CI bootstrap terpisah (III.I.2).

In [12]:
# ==========================================================
# CELL 22: UJI SIGNIFIKANSI -- WILCOXON+HOLM-BONFERRONI (5 HIPOTESIS) + CI QWK
# ==========================================================
from src.significance import (
    collect_all_predictions, aggregate_errors_across_seeds,
    run_significance_test, run_all_effect_sizes, run_all_qwk_effect_sizes,
)
from src import config

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Wilcoxon Signed-Rank (H1-H5) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size MAE + CI 95% (pelengkap Wilcoxon)...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)
effect_sizes.to_csv(os.path.join(config.RESULTS_DIR, "effect_sizes.csv"), index=False)

print("\n📐 Selisih QWK + CI 95% bootstrap (metrik keputusan utama, TANPA Wilcoxon)...")
qwk_effect_sizes = run_all_qwk_effect_sizes(true_labels, preds_per_seed)
display(qwk_effect_sizes)
qwk_effect_sizes.to_csv(os.path.join(config.RESULTS_DIR, "qwk_effect_sizes.csv"), index=False)

📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...
   Memuat prediksi M1_Baseline_CE | seed 42 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Memuat prediksi M1_Baseline_CE | seed 123 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Memuat prediksi M1_Baseline_CE | seed 2024 ...


...

,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.700630,0.761204,2.983634e-12,1.491817e-11,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.722456,0.700630,4.259827e-03,8.519654e-03,Ya
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.723273,0.700630,2.462054e-03,7.386163e-03,Ya
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.722456,0.723273,9.919289e-01,9.919289e-01,Tidak
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,0.727124,0.761204,1.011453e-05,4.045814e-05,Ya



📐 Effect size MAE + CI 95% (pelengkap Wilcoxon)...


,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,-0.060574,-0.075864,-0.045743
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.021825,0.009454,0.033847
2,M5_CleanedHard_CORN,M4_Baseline_CORN,0.022642,0.008754,0.036064
3,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,-0.000817,-0.014122,0.013658
4,M3_CleanedSevere_CE,M1_Baseline_CE,-0.034080,-0.049023,-0.019725



📐 Selisih QWK + CI 95% bootstrap (metrik keputusan utama, TANPA Wilcoxon)...


,model_a,model_b,qwk_a,qwk_b,qwk_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,0.735904,0.734088,0.001815,-0.007278,0.010757
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.725690,0.735904,-0.010213,-0.018618,-0.001387
2,M5_CleanedHard_CORN,M4_Baseline_CORN,0.726271,0.735904,-0.009632,-0.019049,-0.000521
3,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.725690,0.726271,-0.000581,-0.010394,0.008885
4,M3_CleanedSevere_CE,M1_Baseline_CE,0.725385,0.734088,-0.008703,-0.018097,0.000561


## Ringkasan Akhir — Siap Disalin ke Bab 1 (Pilot Study) dan Bab 4

Semua dibaca dari file di Drive (bukan variabel session), jadi cell ini bisa
dijalankan kapan saja, tidak peduli cell mana yang sudah/belum jalan di sesi
Colab saat ini.

In [ ]:
# ==========================================================
# CELL 24: RINGKASAN AKHIR
# ==========================================================
import os
import pandas as pd
from src import config

print("=" * 70)
print(" RINGKASAN LENGKAP ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (P1-P4) -- untuk Bab 1 (pilot study) & Bab 4:")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    display(pd.read_csv(config.PROXY_QUALITY_LOG_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 12 dulu.")

print("\n[2] LAPORAN KUALITAS DATA SCRAPING -- untuk Bab 3:")
for fname in ["scraping_summary.csv", "date_distribution_per_app_rating.csv",
              "text_length_summary.csv", "duplicate_review_texts.csv"]:
    fpath = os.path.join(config.RESULTS_DIR, fname)
    status = "✅" if os.path.exists(fpath) else "⚠️ belum ada"
    print(f"   {status} {fpath}")

print("\n[3] VALIDASI MANUSIA -- untuk Bab 4:")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    df_human = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
    counts = df_human["human_verdict"].value_counts()
    total = len(df_human)
    print(f"   Total: {total} | Noise: {counts.get('noise',0)} ({counts.get('noise',0)/total*100:.1f}%) "
          f"| Not noise: {counts.get('not_noise',0)} ({counts.get('not_noise',0)/total*100:.1f}%) "
          f"| Ambiguous: {counts.get('ambiguous',0)} ({counts.get('ambiguous',0)/total*100:.1f}%)")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 18 dulu.")

print("\n[4] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:")
if os.path.exists(config.FINAL_RESULTS_TABLE_FILE):
    display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 20 dulu.")

print("\n[5] UJI SIGNIFIKANSI (4 hipotesis pre-registered) -- untuk Bab 4:")
if os.path.exists(config.SIGNIFICANCE_TEST_FILE):
    display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n[6] EFFECT SIZE + CI 95% -- untuk Bab 4:")
effect_size_file = os.path.join(config.RESULTS_DIR, "effect_sizes.csv")
if os.path.exists(effect_size_file):
    display(pd.read_csv(effect_size_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n[7] EFFECT SIZE QWK + CI 95% -- untuk Bab 4:")
qwk_file = os.path.join(config.RESULTS_DIR, "qwk_effect_sizes.csv")
if os.path.exists(qwk_file):
    display(pd.read_csv(qwk_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)